In [67]:
import re
import ast
import json
import pandas as pd
from pathlib import Path

In [4]:
EC35_DIR = Path.cwd().parent / "multimodal_er" / "EmoComics35"
DATA_FILES_DIR = EC35_DIR / "data_files"

In [5]:
DATA_FILE = DATA_FILES_DIR / "/Utilisateurs/umushtaq/multimodal_er/EmoComics35/data_files/emocomics35_pg_images.csv"

In [9]:
df = pd.read_csv(DATA_FILE, index_col=0)

In [10]:
df

,SourceFile,Page,PageUtterances,PageSpeakers,PageEmotions,PagePanels,PageBalloons,FileNr,Split,ComicBookTitle,image_path
0,QC copy - 1499 - 58 ECC Co_mics 50 _The Jurass...,1,['THIS VILE THING ATTACKED THE SMALL BEASTS OF...,"['AQUANYX', 'AQUANYX', 'ID-1', 'ID-1', 'AQUANY...","[""['Anger']"", ""['Anger']"", ""['Fear']"", ""['Fear...","[1, 1, 1, 2, 3, 3, 3, 4, 5, 6]","[2, 3, 4, 1, 1, 2, 3, 1, 2, 1]",1499,TRAIN,Jurassic League #4,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
1,QC copy - 1499 - 58 ECC Co_mics 50 _The Jurass...,2,"['NO-- #GKKK…#', '#CHOMP!', 'BY THE SKIN OF M...","['ID-1', 'BLACKMANTASAURUS', 'AQUANYX', 'AQUAN...","[""['Fear']"", ""['Anger']"", ""['Surprise']"", ""['A...","[1, 1, 2, 3, 3, 3, 3, 3, 3]","[1, 2, 1, 1, 2, 3, 5, 6, 7]",1499,TRAIN,Jurassic League #4,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
2,QC copy - 1499 - 58 ECC Co_mics 50 _The Jurass...,3,"['COME ON, BEAST!', 'SHOW YOURSELF!', 'WHY DO ...","['AQUANYX', 'AQUANYX', 'AQUANYX', 'AQUANYX']","[""['Joy']"", ""['Joy']"", ""['Anger']"", ""['Anger']""]","[1, 1, 1, 1]","[1, 2, 5, 6]",1499,TRAIN,Jurassic League #4,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
3,QC copy - 1499 - 58 ECC Co_mics 50 _The Jurass...,4,['#AARGH! '],['AQUANYX'],"[""['Fear', 'Surprise']""]",[2],[2],1499,TRAIN,Jurassic League #4,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
4,QC copy - 1499 - 58 ECC Co_mics 50 _The Jurass...,5,"['I, THE GREEN TORCH, HAVE BEEN TASKED WITH PR...","['GREEN TORCH', 'GREEN TORCH', 'ATROCITAURUS',...","[""['Anger']"", ""['Anger']"", ""['Fear']"", ""['Fear...","[1, 1, 1, 3, 4, 5]","[2, 3, 5, 1, 1, 2]",1499,TRAIN,Jurassic League #4,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
...,...,...,...,...,...,...,...,...,...,...,...
869,QC copy - 2200 - Stillwater 13.xlsx,16,"[""WE WERE IN GALEN'S OFFICE. YOU WERE ABOUT TO...","['LAURA', 'LAURA', 'LAURA', 'DANIEL', 'DANIEL'...","[""['Anger']"", ""['Anger']"", ""['Anger']"", ""['Ang...","[1, 1, 1, 2, 2, 3, 3, 3, 3, 4, 4, 5, 5, 6]","[1, 2, 3, 1, 2, 2, 3, 4, 5, 1, 2, 1, 2, 1]",2200,TEST,Stillwater #13,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
870,QC copy - 2200 - Stillwater 13.xlsx,17,"['SO WHAT ARE WE GOING TO DO?', 'THE WAY I SEE...","['ID-6', 'GALEN', 'ID-7', 'GALEN', 'GALEN', 'G...","[""['Sadness', 'Surprise']"", ""['Anger']"", ""['An...","[3, 3, 3, 3, 4, 4, 5]","[1, 2, 3, 4, 1, 2, 1]",2200,TEST,Stillwater #13,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
871,QC copy - 2200 - Stillwater 13.xlsx,18,"[""KIDDIE COUNCIL'S BEEN GOING A LONG TIME... ""...","['TED', 'KREEGS', 'ID-8', 'ID-8', 'GALEN', 'GA...","[""['Anger', 'Sadness']"", ""['Anger']"", ""['Anger...","[1, 1, 1, 2, 3, 4, 5, 6, 6, 7, 7]","[1, 2, 3, 1, 1, 1, 1, 1, 2, 1, 2]",2200,TEST,Stillwater #13,/Utilisateurs/umushtaq/multimodal_er/EmoComics...
872,QC copy - 2200 - Stillwater 13.xlsx,19,"[""IT'S BEEN… PEACEFUL. ASIDE FROM SHIT LIKE TH...","['KREEGS', 'GALEN', 'GALEN', 'KREEGS', 'GALEN'...","[""['Anger']"", ""['Joy']"", ""['Joy']"", ""['Anger',...","[1, 1, 1, 2, 2, 3, 4, 4, 5, 5, 6, 6]","[1, 2, 3, 1, 2, 1, 1, 2, 2, 3, 1, 2]",2200,TEST,Stillwater #13,/Utilisateurs/umushtaq/multimodal_er/EmoComics...


In [12]:
df_train = df[df.Split == "TRAIN"].reset_index(drop=True)
df_test = df[df.Split == "TEST"].reset_index(drop=True)

### Build Prompts

In [292]:
def generation_instruction():
    
    
    emotion_classes = ["anger", "disgust", "fear", "sadness", "surprise", "joy", "neutral"]
    formatted_classes = ", ".join([f'"{emotion}"' for emotion in emotion_classes])

    instruction = f"""Multimodal Comic Book Emotion Analysis Expert Role

    Task Overview:
    You are an advanced multimodal emotion analysis expert specializing in interpreting comic book page emotions through both visual and textual cues.

    INPUT MODALITIES:
    Visual Input:
    - A full-page image of a comic book page
    - Includes panels, character expressions, visual context, and scene composition

    Textual Input:
    - Numbered list of utterances corresponding to the page

    TASK: Emotion Classification

    - Analyze visual cues in the comic page: Character facial expressions, Body language, Color palette, Panel composition, Character positioning
    - Cross-reference visual emotions with textual utterances
    - Consider contextual and subtle emotional nuances
    - Identify applicable emotions from the following classes:
    {formatted_classes}

    RULES:
    - Use ONLY the labels listed above
    - Output MUST BE a one-line compact JSON with single key "emotions"
    - Respond ONLY with the JSON object. No additional text before or after.
    - Value must be an array where:
    - Each element is an array of emotions for one utterance
    - Order matches the input utterances order
    - Multiple emotions are allowed per utterance
    - No explanations, only JSON output

    IMPORTANT:
    - Respond with a ONE-LINE JSON
    - Each array element corresponds to one utterance
    - One utterance can have multiple emotions
    - Maintain exact spelling and case of emotion labels
    - Keep emotions in arrays even for single emotions
    
    - Example output format: {{"emotions": [["anger", "sadness"], ["joy"], ["fear"], ["neutral"]]}}

    """

    return instruction


In [293]:
def build_input(utterances):
    
    gen_instruction = generation_instruction()
    pg_utterances = "\n".join(f"{i+1}. {s}" for i, s in enumerate(eval(utterances)))
    
    return gen_instruction + "Here are the utterances on the comic page that you will classify:\n" + pg_utterances

In [294]:
def build_output(output):
    
    fixed_data = [item.replace("Neutral", '"Neutral"') for item in eval(output)]
    output = [ast.literal_eval(elem) for elem in fixed_data]
    
    return {"emotions": output}

In [295]:
def build_convo(utterances, output, path):
  
  
    
  conversation = {
    
  "conversations": [
    {
      "from": "human",
      "value": "<image>" + build_input(utterances)
    },
    {
      "from": "gpt",
      "value": build_output(output)
    }
  ],
  "images": [
    path
  ]

  }

  return conversation

### Data Files

In [296]:
data_train_l = []

for idx, row in df_train.iterrows():

    try:
        conversation = build_convo(row.PageUtterances, row.PageEmotions, row.image_path)
        data_train_l.append(conversation)
    except Exception as e:
        print(f"Error at index {idx}: {e}")
    #break

In [297]:
data_train_l[5]

{'conversations': [{'from': 'human',
   'value': '<image>Multimodal Comic Book Emotion Analysis Expert Role\n\n    Task Overview:\n    You are an advanced multimodal emotion analysis expert specializing in interpreting comic book page emotions through both visual and textual cues.\n\n    INPUT MODALITIES:\n    Visual Input:\n    - A full-page image of a comic book page\n    - Includes panels, character expressions, visual context, and scene composition\n\n    Textual Input:\n    - Numbered list of utterances corresponding to the page\n\n    TASK: Emotion Classification\n\n    - Analyze visual cues in the comic page: Character facial expressions, Body language, Color palette, Panel composition, Character positioning\n    - Cross-reference visual emotions with textual utterances\n    - Consider contextual and subtle emotional nuances\n    - Identify applicable emotions from the following classes:\n    "anger", "disgust", "fear", "sadness", "surprise", "joy", "neutral"\n\n    RULES:\n    

In [298]:
data_test_l = []

for idx, row in df_test.iterrows():

    try:
        conversation = build_convo(row.PageUtterances, row.PageEmotions, row.image_path)
        data_test_l.append(conversation)
    except Exception as e:
        print(f"Error at index {idx}: {e}")

In [299]:
data_test_l[0]

{'conversations': [{'from': 'human',
   'value': '<image>Multimodal Comic Book Emotion Analysis Expert Role\n\n    Task Overview:\n    You are an advanced multimodal emotion analysis expert specializing in interpreting comic book page emotions through both visual and textual cues.\n\n    INPUT MODALITIES:\n    Visual Input:\n    - A full-page image of a comic book page\n    - Includes panels, character expressions, visual context, and scene composition\n\n    Textual Input:\n    - Numbered list of utterances corresponding to the page\n\n    TASK: Emotion Classification\n\n    - Analyze visual cues in the comic page: Character facial expressions, Body language, Color palette, Panel composition, Character positioning\n    - Cross-reference visual emotions with textual utterances\n    - Consider contextual and subtle emotional nuances\n    - Identify applicable emotions from the following classes:\n    "anger", "disgust", "fear", "sadness", "surprise", "joy", "neutral"\n\n    RULES:\n    

### JSON Files

In [300]:
file_path = Path(EC35_DIR) / "json_datasets" / "EC35_mm_pg_train.json"

with open(file_path, 'w') as file:
    
    json.dump(data_train_l, file)

In [301]:
file_path = Path(EC35_DIR) / "json_datasets" /"EC35_mm_pg_test.json"

with open(file_path, 'w') as file:
    
    json.dump(data_test_l, file)